# Step 2 — Full PDE residual via a differentiable PFC solver

This notebook trains the FNO **with** and **without** the step-2 physics term
(`pde_weight`) and produces the diagnostics to judge it. The term enforces the
*whole* PFC equation, not just the energy inequality of step 1: a differentiable
copy of the solver advances the model's input frame and the prediction is
penalized for disagreeing. During rollout this is evaluated on the model's own
**unlabeled** states, so it regularizes drift where the data loss is silent.
The solver runs **only during training** — inference speed is unchanged.

**What this notebook gives you**
1. A correctness gate: `verify_pde_step.py` confirms the torch stepper matches
   the numpy solver (otherwise the residual is meaningless).
2. Data generation (regenerates the grain-rich dataset if it is not cached).
3. An ensembled A/B: `pde_weight = 0` (baseline) vs `0.1` (step 2), 2 seeds each,
   because we know single runs are dominated by training noise.
4. Success metrics (rollout MSE, phase-insensitive spectral MSE) **and** the new
   PDE-residual diagnostic that pinpoints *where and when* a rollout leaves the
   physical manifold — even on frames with no ground truth.

Set the runtime: **Runtime → Change runtime type → T4 GPU**, then **Run all**.

### 1. Mount Drive + clone repo + install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/Tntguy0128/crystal-growth-sim.git
%cd crystal-growth-sim
!pip install -q torch numpy scipy pyyaml matplotlib tensorboard

### 2. Confirm a GPU is attached

In [ ]:
import torch
assert torch.cuda.is_available(), (
    "NO GPU. Runtime -> Change runtime type -> T4 GPU, then Run all again.")
print("GPU OK:", torch.cuda.get_device_name(0))

### 3. Gate: verify the differentiable solver matches the numpy solver
The PDE residual is only meaningful if `DifferentiablePFCStep` reproduces
`pfc_solver.py`. This must print **PASS** before we trust any step-2 result.

In [ ]:
import subprocess, os
env = {**os.environ, 'PYTHONUNBUFFERED': '1'}
res = subprocess.run(['python', '-u', 'verify_pde_step.py'], env=env)
assert res.returncode == 0, "Stepper verification FAILED — do not trust step-2 results."

### 4. Get the grain-rich dataset
Reuses `pfc_dataset_grains.zip` from Drive if present (instant); otherwise
generates it from `pfc_config_grains.yaml` and caches it back to Drive.

In [ ]:
import os
drive_zip = '/content/drive/MyDrive/pfc_dataset_grains.zip'
if os.path.exists(drive_zip):
    print('Found dataset on Drive -- unzipping.')
    !cp "$drive_zip" .
    !unzip -o -q pfc_dataset_grains.zip
else:
    print('No dataset on Drive -- generating it now.')
    !python generate_dataset.py --sweep --config pfc_config_grains.yaml --parallel --max-workers 2
    !zip -r -q pfc_dataset_grains.zip data_pfc_grains
    !cp pfc_dataset_grains.zip /content/drive/MyDrive/
import pandas as pd
m = pd.read_csv('data_pfc_grains/manifest.csv')
print(m['status'].value_counts().to_dict(), '|', len(m), 'runs')

### 5. Train the A/B ensemble
Two arms (`pde_weight` 0.0 vs 0.1), `SEEDS` runs each, all on the same dataset
and `split_seed` so the test sets match and the comparison is apples-to-apples.
Each model + eval is copied to Drive as it finishes (so a disconnect loses at
most one run). Trim `SEEDS` to `[0]` for a quick single-run look.

In [ ]:
import yaml, subprocess, shutil, os

SEEDS = [0, 1]                       # ensemble per arm (set [0] for a quick look)
ARMS  = {'baseline': 0.0, 'pde': 0.1}   # tag -> pde_weight
env = {**os.environ, 'PYTHONUNBUFFERED': '1'}

def run(tag, pde_w, seed):
    run_tag = f'{tag}_s{seed}'
    with open('config.yaml') as f:
        cfg = yaml.safe_load(f)
    cfg['data']['data_dir'] = 'data_pfc_grains'
    cfg['data']['split_by'] = 'config'
    cfg['train']['physics_weight'] = 0.0     # isolate step 2: step-1 penalty OFF
    cfg['train']['pde_weight']     = pde_w
    cfg['train']['seed']           = seed
    out_dir = f'runs/pde_{run_tag}'
    cfg['logging']['out_dir'] = out_dir
    cfg['eval']['out_dir']    = f'{out_dir}/eval'
    with open('config.yaml', 'w') as f:
        yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)

    print(f'===== TRAIN [{run_tag}]  pde_weight={pde_w}  seed={seed} =====', flush=True)
    subprocess.run(['python', '-u', 'train_fno.py', '--config', 'config.yaml'],
                   check=True, env=env)
    ckpt = f'{out_dir}/best.pt'
    shutil.copy(ckpt, f'/content/drive/MyDrive/fno_pde_{run_tag}.pt')
    print(f'===== EVAL [{run_tag}] =====', flush=True)
    subprocess.run(['python', '-u', 'evaluate_fno.py', '--config', 'config.yaml',
                    '--checkpoint', ckpt], check=True, env=env)
    !zip -r -q fno_pde_eval_{run_tag}.zip runs/pde_{run_tag}/eval
    !cp fno_pde_eval_{run_tag}.zip /content/drive/MyDrive/
    return out_dir

RUNS = {}
for tag, pde_w in ARMS.items():
    for seed in SEEDS:
        RUNS[f'{tag}_s{seed}'] = run(tag, pde_w, seed)
print('\nAll runs complete:', list(RUNS))

### 6. Success metrics — did step 2 help?
Per-run checkpoint metadata + held-out rollout / spectral MSE, then the arm
means ± spread across seeds. Spectral MSE is the phase-insensitive metric
(a benign translation scores ~0; a wrong grain orientation scores high), so a
gap between rollout and spectral MSE separates real structural errors from
harmless phase offsets.

In [ ]:
import pandas as pd, numpy as np, torch
rows = []
for run_tag, out_dir in RUNS.items():
    df = pd.read_csv(f'{out_dir}/eval/per_trajectory.csv')
    ck = torch.load(f'{out_dir}/best.pt', map_location='cpu', weights_only=False)
    arm = run_tag.rsplit('_s', 1)[0]
    rows.append(dict(run=run_tag, arm=arm,
                     pde_weight=ck['config']['train'].get('pde_weight', 0.0),
                     best_epoch=ck.get('epoch'),
                     best_val=round(float(ck.get('val_loss')), 6),
                     rollout_mse=round(float(df['rollout_mse'].mean()), 6),
                     spectral_mse=round(float(df['spectral_mse'].mean()), 6),
                     phase_offset=int(df['phase_offset'].sum())))
res = pd.DataFrame(rows).sort_values(['arm', 'run'])
print('Per-run:'); display(res)
print('\nArm means +/- std across seeds:')
display(res.groupby('arm')[['rollout_mse', 'spectral_mse']].agg(['mean', 'std']))

### 7. Per-seed-type breakdown
Step 1 was *regime-dependent* (helped multi-grain coarsening, hurt clean
hex/point). This shows whether step 2 has the same signature or a different one.

In [ ]:
frames = []
for run_tag, out_dir in RUNS.items():
    d = pd.read_csv(f'{out_dir}/eval/per_trajectory.csv')[['seed_type', 'rollout_mse']]
    d = d.groupby('seed_type')['rollout_mse'].mean().rename(run_tag)
    frames.append(d)
table = pd.concat(frames, axis=1)
print('mean rollout MSE by seed_type (columns = runs):')
display(table)

### 8. PDE-residual diagnostic — *where and when* does the rollout break?
This is the payoff of the differentiable solver. For one model per arm (seed 0)
we roll out the held-out trajectories and, at every step, measure the relative
PDE residual between the prediction and the solver applied to the previous
prediction. It needs **no ground truth**, so it scores the unlabeled states the
model drifts into, and its spatial map points at the exact grains that go wrong.
We overlay the two arms: if step 2 works, its residual curve sits **below**
the baseline (the rollout stays closer to the physics manifold).

In [ ]:
import matplotlib.pyplot as plt
from pde_diagnostics import run_pde_diagnostics
from IPython.display import Image, display

diag = {}
for arm in ARMS:                       # one representative seed per arm
    out_dir = RUNS[f'{arm}_s{SEEDS[0]}']
    diag[arm] = run_pde_diagnostics(f'{out_dir}/best.pt',
                                    out_dir=f'{out_dir}/pde_diag',
                                    max_traj=8, label=arm)

# Overlay the two residual-vs-step curves.
plt.figure(figsize=(7, 4))
for arm, d in diag.items():
    plt.plot(d['steps'], d['mean_curve'], '-o', lw=2, label=arm)
    plt.fill_between(d['steps'], d['mean_curve'] - d['std_curve'],
                     d['mean_curve'] + d['std_curve'], alpha=0.15)
plt.xlabel('rollout step'); plt.ylabel('relative PDE residual')
plt.title('Off-manifold drift: baseline vs step 2 (lower is better)')
plt.grid(alpha=0.3); plt.legend(); plt.tight_layout()
plt.savefig('runs/pde_residual_overlay.png', dpi=120); plt.show()

# Worst-frame spatial maps for each arm (where the physics breaks).
for arm, d in diag.items():
    print(f'--- {arm}: worst physics violation ---')
    display(Image(d['map_png']))

### 9. Worst held-out rollouts (the qualitative failures)

In [ ]:
import glob
for arm in ARMS:
    out_dir = RUNS[f'{arm}_s{SEEDS[0]}']
    print(f'================  {arm}  ================')
    for png in sorted(glob.glob(f'{out_dir}/eval/*.png'))[:2]:
        display(Image(png))

### 10. Save all diagnostics to Drive

In [ ]:
!cp runs/pde_residual_overlay.png /content/drive/MyDrive/
for arm in ARMS:
    out_dir = RUNS[f'{arm}_s{SEEDS[0]}']
    !zip -r -q pde_diag_{arm}.zip {out_dir}/pde_diag
    !cp pde_diag_{arm}.zip /content/drive/MyDrive/
res.to_csv('/content/drive/MyDrive/pde_step2_summary.csv', index=False)
print('Saved to Drive: pde_step2_summary.csv, pde_residual_overlay.png,',
      'pde_diag_<arm>.zip, fno_pde_<run>.pt, fno_pde_eval_<run>.zip')